# Idealista Barcelona — Detail Scraper (Stealth)

Reads `data/idealista_barcelona_sale_urls.csv`, fetches each listing page, and appends rows to `data/idealista_barcelona_sale_properties_details.csv`.

**Anti-detection stack:**
- `undetected-chromedriver` patches the Chrome binary so `navigator.webdriver` is hidden
- Persistent Chrome profile in `data/chrome_profile` keeps real cookies across sessions
- Random delays 15–45 s between pages; every 50 pages the browser restarts
- Random scroll + micro-pause before extracting, mimicking human reading
- `HEADLESS = False` so Idealista's canvas/WebGL fingerprinting sees a real GPU context

**Resuming:** the script skips any URL already present in the output CSV, so you can stop/restart freely.

In [ ]:
# %pip install undetected-chromedriver beautifulsoup4 pandas tqdm lxml

In [ ]:
import csv
import hashlib
import json
import random
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

In [ ]:
# ── paths ────────────────────────────────────────────────────────────────────
DATA_DIR        = Path("data")
URLS_CSV        = DATA_DIR / "idealista_barcelona_sale_urls.csv"
OUTPUT_CSV      = DATA_DIR / "idealista_barcelona_sale_properties_details.csv"
CACHE_DIR       = DATA_DIR / "html_cache" / "detail_pages"
PROFILE_DIR     = DATA_DIR / "chrome_profile"   # persistent cookies / login state
for d in [CACHE_DIR, PROFILE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── timing ───────────────────────────────────────────────────────────────────
MIN_DELAY            = 15      # seconds between detail pages
MAX_DELAY            = 45
LONG_BREAK_EVERY     = 50      # restart browser + take a longer break every N pages
LONG_BREAK_MIN       = 120     # seconds for the long break
LONG_BREAK_MAX       = 240

# ── run controls ─────────────────────────────────────────────────────────────
MAX_DETAILS          = None    # set to e.g. 25 for a test run; None = all
USE_HTML_CACHE       = True    # skip re-fetching pages that are already cached
WAIT_ON_BLOCK        = True    # pause so you can solve a CAPTCHA in the open window
BLOCK_WAIT_SECONDS   = 300
LOG_EVERY            = 5       # print a status line every N pages

COLUMNS = [
    "propertyCode", "Link", "district", "neighborhood",
    "price", "size", "bed", "br", "floor",
    "address", "latitude", "longitude", "x", "y",
    "url", "description", "scraped_at",
]

In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────────

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)


def now_utc_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def clean(v):
    if v is None:
        return None
    v = re.sub(r"\s+", " ", str(v)).strip()
    return v or None


def as_number(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return v
    m = re.search(r"-?\d+(?:[.,]\d+)?", str(v).replace(".", ""))
    if not m:
        return None
    n = float(m.group(0).replace(",", "."))
    return int(n) if n.is_integer() else n


def property_code(url):
    m = re.search(r"/inmueble/(\d+)/", str(url))
    return m.group(1) if m else None


def cache_path(url):
    digest = hashlib.sha1(url.encode()).hexdigest()[:16]
    code = property_code(url)
    stem = f"{code}_{digest}" if code else digest
    return CACHE_DIR / f"{stem}.html"


def load_cache(url):
    p = cache_path(url)
    if USE_HTML_CACHE and p.exists():
        return p.read_text(encoding="utf-8")
    return None


def save_cache(url, html):
    if USE_HTML_CACHE and html:
        cache_path(url).write_text(html, encoding="utf-8")

In [ ]:
# ── browser ───────────────────────────────────────────────────────────────────

def make_driver():
    """Spin up an undetected Chrome instance with a persistent profile."""
    options = uc.ChromeOptions()
    options.add_argument(f"--user-data-dir={PROFILE_DIR.resolve()}")
    options.add_argument("--window-size=1400,900")
    options.add_argument("--lang=en-US,en")
    # NOT headless: canvas/WebGL fingerprinting needs a real GPU context
    driver = uc.Chrome(options=options, use_subprocess=True)
    driver.set_page_load_timeout(60)
    log("Chrome ready")
    return driver


def quit_driver(driver):
    try:
        driver.quit()
    except Exception:
        pass


def accept_cookies(driver):
    for label in ["Accept all", "Accept", "Aceptar todas", "Aceptar"]:
        try:
            for btn in driver.find_elements(By.XPATH, f"//button[contains(normalize-space(.), '{label}')]"):
                if btn.is_displayed() and btn.is_enabled():
                    btn.click()
                    time.sleep(1.5)
                    return
        except WebDriverException:
            pass


def is_blocked(driver):
    try:
        body = driver.find_element(By.TAG_NAME, "body").text.lower()
        title = (driver.title or "").lower()
    except WebDriverException:
        return False
    visible = title + " " + body
    hard = ["unusual traffic", "verify you are human", "access denied",
            "checking your browser", "se ha detectado un uso indebido",
            "el acceso se ha bloqueado"]
    if any(m in visible for m in hard):
        return True
    has_listing = "/inmueble/" in body or "property for sale" in body
    return "captcha" in visible and not has_listing


def handle_block(driver, url):
    if not WAIT_ON_BLOCK:
        raise RuntimeError("Blocked by Idealista. Stop scraping for now.")
    log("Block/CAPTCHA detected — solve it in the Chrome window, then press Enter here")
    input("Press Enter once the page looks normal: ")
    if not is_blocked(driver):
        log("Unblocked — continuing")
        return
    deadline = time.time() + BLOCK_WAIT_SECONDS
    while time.time() < deadline:
        time.sleep(10)
        if not is_blocked(driver):
            log("Unblocked — continuing")
            return
    raise RuntimeError(f"Still blocked after {BLOCK_WAIT_SECONDS}s on {url}")


def human_scroll(driver):
    """Scroll down the page in a few steps, like a person reading."""
    try:
        height = driver.execute_script("return document.body.scrollHeight")
        steps = random.randint(3, 6)
        for i in range(1, steps + 1):
            target = int(height * i / steps)
            driver.execute_script(f"window.scrollTo(0, {target});")
            time.sleep(random.uniform(0.4, 1.2))
        # scroll back up a bit
        driver.execute_script(f"window.scrollTo(0, {random.randint(0, 300)});")
        time.sleep(random.uniform(0.3, 0.8))
    except WebDriverException:
        pass


def fetch_page(driver, url):
    """Load a URL and return HTML, using cache when available."""
    cached = load_cache(url)
    if cached is not None:
        return cached

    try:
        driver.get(url)
    except TimeoutException:
        try:
            driver.execute_script("window.stop();")
        except WebDriverException:
            pass
    except WebDriverException as exc:
        log(f"Failed to load {url}: {exc}")
        return None

    accept_cookies(driver)

    if is_blocked(driver):
        handle_block(driver, url)
        accept_cookies(driver)

    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
    except TimeoutException:
        pass

    human_scroll(driver)

    try:
        html = driver.page_source
    except WebDriverException:
        return None

    save_cache(url, html)
    return html

In [ ]:
# ── parsing ───────────────────────────────────────────────────────────────────

def json_walk(obj):
    if isinstance(obj, dict):
        yield obj
        for v in obj.values():
            yield from json_walk(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from json_walk(v)


def load_jsonld(soup):
    out = []
    for tag in soup.select("script[type='application/ld+json']"):
        raw = tag.string or tag.get_text()
        if not raw:
            continue
        try:
            out.extend(json_walk(json.loads(raw)))
        except Exception:
            pass
    return out


def schema_value(objects, *keys):
    keys = {k.lower() for k in keys}
    for obj in objects:
        for k, v in obj.items():
            if k.lower() in keys and v not in (None, "", []):
                return v
    return None


def schema_address(objects):
    v = schema_value(objects, "address")
    if isinstance(v, dict):
        parts = [v.get(k) for k in ["streetAddress", "addressLocality", "addressRegion", "postalCode"]]
        return clean(", ".join(str(x) for x in parts if x))
    return clean(v)


def schema_geo(objects):
    for obj in objects:
        if "latitude" in obj and "longitude" in obj:
            return obj["latitude"], obj["longitude"]
        geo = obj.get("geo")
        if isinstance(geo, dict) and "latitude" in geo:
            return geo["latitude"], geo["longitude"]
    return None, None


def regex_geo(html):
    # JSON-style lat/lon
    patterns = [
        r'"latitude"\s*:\s*([\-\d.]+)\s*,\s*"longitude"\s*:\s*([\-\d.]+)',
        r'"longitude"\s*:\s*([\-\d.]+)\s*,\s*"latitude"\s*:\s*([\-\d.]+)',
    ]
    for i, pat in enumerate(patterns):
        m = re.search(pat, html, re.I | re.S)
        if m:
            return (m.group(2), m.group(1)) if i == 1 else (m.group(1), m.group(2))
    # Google Static Maps URL: center=41.3922%2C2.1754
    m = re.search(r'center=([\-\d.]+)(?:%2C|,)([\-\d.]+)', html, re.I)
    if m:
        return m.group(1), m.group(2)
    return None, None


def visible(soup, *selectors):
    for sel in selectors:
        node = soup.select_one(sel)
        if node:
            t = clean(node.get_text(" ", strip=True))
            if t:
                return t
    return None


def parse_features(text):
    text = clean(text) or ""
    out = {"size": None, "bed": None, "br": None, "floor": None}
    m = re.search(r"([\d.,]+)\s*m[²2]", text, re.I)
    if m:
        out["size"] = as_number(m.group(1))
    m = re.search(r"(\d+)\s*bed", text, re.I)
    if m:
        out["bed"] = int(m.group(1))
    m = re.search(r"(\d+)\s*bath", text, re.I)
    if m:
        out["br"] = int(m.group(1))
    for pat in [r"(\d+)(?:st|nd|rd|th)?\s*floor", r"floor\s*(\d+)",
                r"(ground floor|basement|semi-basement|mezzanine|top floor)"]:
        m = re.search(pat, text, re.I)
        if m:
            out["floor"] = m.group(1).lower()
            break
    return out


def parse_district_neighborhood(address):
    parts = [clean(x) for x in str(address or "").split(",")]
    parts = [x for x in parts if x]
    if len(parts) >= 3:
        return parts[-1].replace("Barcelona", "").strip() or None, parts[-2]
    if len(parts) == 2:
        return None, parts[-1]
    return None, None


def detail_features_text(soup, fallback=None):
    chunks = [x.get_text(" ", strip=True) for x in soup.select(
        ".details-property_features li, .info-features span, "
        ".details-property-feature-one, .details-property-feature-two, .item-detail"
    )]
    return clean(" | ".join(chunks)) or fallback


def parse_detail(html, url, search_row):
    soup = BeautifulSoup(html, "lxml")
    objects = load_jsonld(soup)
    features = parse_features(detail_features_text(soup, search_row.get("details_search")))

    lat, lon = schema_geo(objects)
    if not lat:
        lat, lon = regex_geo(html)

    address = (
        schema_address(objects)
        or visible(soup, "span.main-info__title-main", ".main-info__title-main", "h1")
        or search_row.get("address_search")
    )
    district, neighborhood = parse_district_neighborhood(address)

    price = (
        clean(schema_value(objects, "price"))
        or visible(soup, "span.info-data-price", ".info-data-price", "span.item-price")
        or search_row.get("price_search")
    )

    description = (
        clean(schema_value(objects, "description"))
        or visible(soup, "div.comment", ".adCommentsLanguage", "#details .comment",
                   "[class*='description']")
        or search_row.get("description_search")
    )

    size_schema = schema_value(objects, "floorSize", "size", "area")
    if isinstance(size_schema, dict):
        size_schema = size_schema.get("value") or size_schema.get("amount")

    return {
        "propertyCode": search_row.get("propertyCode") or property_code(url),
        "Link": "LINK",
        "district": district,
        "neighborhood": neighborhood,
        "price": price,
        "size": as_number(size_schema) or features["size"],
        "bed": as_number(schema_value(objects, "numberOfBedrooms", "numberOfRooms")) or features["bed"],
        "br": as_number(schema_value(objects, "numberOfBathroomsTotal", "numberOfBathrooms")) or features["br"],
        "floor": clean(schema_value(objects, "floorLevel", "floor")) or features["floor"],
        "address": address,
        "latitude": lat,
        "longitude": lon,
        "x": lon,
        "y": lat,
        "url": url,
        "description": description,
        "scraped_at": now_utc_iso(),
    }

In [ ]:
# ── main scrape loop ──────────────────────────────────────────────────────────

def load_existing():
    if OUTPUT_CSV.exists():
        df = pd.read_csv(OUTPUT_CSV)
        log(f"Resuming: {len(df):,} rows already in {OUTPUT_CSV}")
        return df.reindex(columns=COLUMNS).to_dict("records"), set(df["url"].dropna().astype(str))
    log("Starting fresh — no existing output CSV found")
    return [], set()


def save(rows):
    pd.DataFrame(rows).reindex(columns=COLUMNS).drop_duplicates("url", keep="last").to_csv(
        OUTPUT_CSV, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL
    )


def run():
    urls_df = pd.read_csv(URLS_CSV).dropna(subset=["url"]).drop_duplicates("url")
    log(f"URLs loaded: {len(urls_df):,}")

    rows, done = load_existing()
    todo = urls_df[~urls_df["url"].astype(str).isin(done)].copy()
    if MAX_DETAILS:
        todo = todo.head(MAX_DETAILS)
    log(f"To scrape: {len(todo):,} | Already done: {len(done):,}")

    driver = make_driver()
    page_count = 0

    try:
        for i, search_row in enumerate(tqdm(todo.to_dict("records"), desc="Detail pages"), start=1):
            url = search_row["url"]
            code = search_row.get("propertyCode") or property_code(url)

            # restart browser every LONG_BREAK_EVERY pages that required a real fetch
            if page_count > 0 and page_count % LONG_BREAK_EVERY == 0:
                secs = random.uniform(LONG_BREAK_MIN, LONG_BREAK_MAX)
                log(f"--- Long break: restarting browser and sleeping {secs:.0f}s ---")
                quit_driver(driver)
                time.sleep(secs)
                driver = make_driver()

            if i % LOG_EVERY == 1:
                log(f"Page {i}/{len(todo)}: {code} | {url}")

            cached = load_cache(url)
            html = cached if cached is not None else fetch_page(driver, url)
            if cached is None:
                page_count += 1  # only count live fetches toward the restart budget

            if not html:
                log(f"Skipping (no HTML): {url}")
                continue

            row = parse_detail(html, url, search_row)
            rows.append(row)
            save(rows)

            if i % LOG_EVERY == 1:
                fields = [f for f in ["price","size","bed","br","latitude"] if row.get(f)]
                log(f"  saved — present: {fields}")

            # human-paced delay only between live fetches
            if cached is None:
                time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

    finally:
        quit_driver(driver)

    final = pd.DataFrame(rows).reindex(columns=COLUMNS).drop_duplicates("url", keep="last")
    log(f"Done: {len(final):,} rows in {OUTPUT_CSV}")
    return final


properties_df = run()
properties_df.head()

In [ ]:
# ── quality check ─────────────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path
OUTPUT_CSV = Path("data/idealista_barcelona_sale_properties_details.csv")

df = pd.read_csv(OUTPUT_CSV)
display(df.head())
display(df.isna().mean().sort_values(ascending=False).to_frame("missing_share"))
print(f"Rows: {len(df):,}")
print(f"Unique property codes: {df['propertyCode'].nunique():,}")
print(f"Rows with lat/lon: {df[['latitude','longitude']].notna().all(axis=1).sum():,}")